# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
To reduce multicollinearity and improve balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


exp_name = "E8_clean_vascular"
H = 5  # horizon years used in build_cohort notebook

OUT_DIR = os.path.join("..", "output")

cohort_file = f"cohort_{exp_name}_H{H}.csv"
DATA_PATH = os.path.join(OUT_DIR, cohort_file)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find cohort file: {DATA_PATH}\n"
        f"Make sure 01_build_cohort exported: {cohort_file}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", cohort_file, df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Loaded: cohort_E8_clean_vascular_H5.csv (38341, 42)
label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64


,NACCID,baseline_date,label,max_follow_years,has_H_followup,has_dementia_H,NACCAGE,SEX,RACE,HISPANIC,...,RIGIDL,RIGIDR,SLOWINGL,SLOWINGR,PARKGAIT,GAITDIS,VASC_RISK_COUNT,STROKE_TIA_ANY,VASC_MED_COUNT,GAIT_ANY
0,NACC000011,2006-04-17,Censored,3.296372,False,False,62.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0,0.0,0
1,NACC000034,2015-07-16,NoDementia_Hy,6.986995,True,False,79.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,1.0,0,2.0,0
2,NACC000067,2012-05-23,Censored,2.088980,False,False,60.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0,1.0,1
3,NACC000073,2016-05-20,Censored,0.000000,False,False,44.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0,0.0,0
4,NACC000138,2023-07-13,Censored,1.043121,False,False,54.0,2.0,50.0,1.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0,0.0,0


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figs_baseline")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [3]:
def fix_categorical_codes(X):
    X = X.copy()
    CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X.columns]
    STD_MISS = {-4, 9}

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(STD_MISS)), np.nan)
        X[c] = s

    if "RACE" in X.columns:
        X["RACE"] = X["RACE"].mask(X["RACE"].isin([99]), np.nan)

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        X[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

    if "EDUC" in X.columns:
        educ_num = pd.to_numeric(X["EDUC"], errors="coerce")
        X["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

    X = X.replace({pd.NA: np.nan})
    for c in X.columns:
        if X[c].dtype == "bool":
            X[c] = X[c].astype(int)
    return X

In [4]:
import numpy as np
import pandas as pd


df4 = df.copy()  

# Keeping only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_Hy", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Dropping IDs/labels
#y = df4["label"].astype(str)
# drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
#drop_cols = [c for c in [ "NACCID", "baseline_date", "label", "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


def prepare_experiment_df(df, exp_name):
    df = df.copy()

    # columns that are always dropped
    drop_cols = [c for c in ["NACCID","baseline_date","label","has_H_followup","has_dementia_H","max_follow_years","abs_days_from_baseline"]
                 if c in df.columns]

    # E8: engineered vascular burden; dropping raw components to avoid collinearity
    if exp_name == "E8_clean_vascular":
        raw_vasc = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
        drop_cols += [c for c in raw_vasc if c in df.columns]

        raw_meds = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
        drop_cols += [c for c in raw_meds if c in df.columns]

        # drop_cols += [c for c in ["GAITDIS"] if c in df.columns]

    # E9: exec ratio; Dropping raw Trails
    if exp_name == "E9_clean_exec_ratio":
        drop_cols += [c for c in ["TRAILA","TRAILB","TRAILA_missing","TRAILB_missing"] if c in df.columns]

    # E10: building exec-memory axis (train-only z-scoring later), then Dropping raw
    # Here, what is dropped is only marked AFTER computing engineered features per split.

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["label"].astype(str)

    return X, y


from sklearn.model_selection import train_test_split

X, y = prepare_experiment_df(df4, exp_name)

# Applying categorical cleanup before splitting
X = fix_categorical_codes(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())
print("Test label counts:\n", y_test.value_counts())



def z_from_train(train_s, s):
    mu = train_s.mean()
    sd = train_s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s*0.0
    return (s - mu) / sd

if exp_name == "E10_exec_mem_axis_raw":
    needed = ["NACCMMSE", "NACCMOCA", "TRAILB"]
    missing = [c for c in needed if c not in X_train.columns]
    if missing:
        raise ValueError(f"E10 missing required columns: {missing}")

    # Z-scores using train stats only
    X_train["Z_TRAILB"] = z_from_train(X_train["TRAILB"], X_train["TRAILB"])
    X_test["Z_TRAILB"]  = z_from_train(X_train["TRAILB"], X_test["TRAILB"])

    mem_cols = [c for c in ["NACCMMSE","NACCMOCA"] if c in X_train.columns]
    Zm_train = []
    Zm_test  = []
    for c in mem_cols:
        Zm_train.append(z_from_train(X_train[c], X_train[c]))
        Zm_test.append(z_from_train(X_train[c], X_test[c]))

    X_train["Z_MEM"] = np.nanmean(np.vstack([z.values for z in Zm_train]), axis=0)
    X_test["Z_MEM"]  = np.nanmean(np.vstack([z.values for z in Zm_test]), axis=0)

    # Executive minus memory axis
    X_train["EXEC_MINUS_MEM"] = X_train["Z_TRAILB"] - X_train["Z_MEM"]
    X_test["EXEC_MINUS_MEM"]  = X_test["Z_TRAILB"]  - X_test["Z_MEM"]

    # Dropping raw components to avoid collinearity
    drop_raw = [
        "TRAILA","TRAILB","NACCMMSE","NACCMOCA",
        "TRAILA_missing","TRAILB_missing","NACCMMSE_missing","NACCMOCA_missing",
        "Z_TRAILB","Z_MEM"
    ]
    X_train = X_train.drop(columns=[c for c in drop_raw if c in X_train.columns])
    X_test  = X_test.drop(columns=[c for c in drop_raw if c in X_test.columns])

    print("E10 engineered EXEC_MINUS_MEM added. Shapes now:", X_train.shape, X_test.shape)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

4-class rows: 14620
label
NoDementia_Hy    11083
AD                3068
PD_LBD             297
Vascular           172
Name: count, dtype: int64
Train: (11696, 37) Test: (2924, 37)
Train label counts:
 label
NoDementia_Hy    8866
AD               2454
PD_LBD            238
Vascular          138
Name: count, dtype: int64
Test label counts:
 label
NoDementia_Hy    2217
AD                614
PD_LBD             59
Vascular           34
Name: count, dtype: int64


In [5]:
csf_cols = ["CSFABETA", "CSFPTAU", "CSFTTAU"]
present_csf_cols = [c for c in csf_cols if c in df.columns]

if len(present_csf_cols) == len(csf_cols):
    df_csf = df[df[present_csf_cols].notna().all(axis=1)].copy()
    print("CSF subset size:", len(df_csf))
    print(df_csf["label"].value_counts())
else:
    print("CSF columns not present in this dataset:", 
          [c for c in csf_cols if c not in df.columns])

CSF columns not present in this dataset: ['CSFABETA', 'CSFPTAU', 'CSFTTAU']


## Split + preprocess + train

Train/test split

In [6]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
# )

Preprocessing pipelines (sparse for LR/boosting)

In [7]:
# cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
# num_cols = [c for c in X_train.columns if c not in cat_cols]

import pandas as pd

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))
print(sorted([c for c in X_train.columns if "VASC" in c or "EXEC" in c or "TRAIL" in c or "MMSE" in c or "MOCA" in c]))
print(sorted([c for c in X_train.columns]))



Num cols: 32 Cat cols: 5
['NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC_MED_COUNT', 'VASC_RISK_COUNT']
['ANIMALS', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'COGFLUC', 'EDUC', 'EDUC_BIN', 'GAITDIS', 'GAIT_ANY', 'HISPANIC', 'NACCAGE', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCUDSD', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SLOWINGL', 'SLOWINGR', 'STROKE_TIA_ANY', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC_MED_COUNT', 'VASC_RISK_COUNT']


In [8]:
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.over_sampling import SMOTE

## The block below has the SMOTE results while the block afterwards is the normal, however the normal has some better results.

In [9]:
# from sklearn.linear_model import LogisticRegression

# custom_w = {
#     "NoDementia_Hy": 1.0,
#     "AD": 2.0,
#     "PD_LBD": 6.0,
#     "Vascular": 8.0
# }

# smote = SMOTE(sampling_strategy="not majority", k_neighbors=2, random_state=42)

# clf_lr = ImbPipeline(steps=[
#     ("preprocess", preprocess),
#     ("smote", smote),
#     ("model", LogisticRegression(max_iter=8000, class_weight=None, C=0.1))
# ])

# clf_lr.fit(X_train, y_train)
# pred_lr = clf_lr.predict(X_test)

# print("=== Logistic Regression (SMOTE, C=0.1) ===")
# print(classification_report(y_test, pred_lr, digits=4))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
# print(confusion_matrix(y_test, pred_lr, labels=labels))


Model A: Logistic Regression (final settings)

# Section 3C: Logistic Regression (balanced weights + vascular boost + threshold)

In [10]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline
# from sklearn.base import clone
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
# import numpy as np

# # 1) Systematic class weights (balanced) + optional vascular boost
# classes = np.unique(y_train)

# base_w = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=y_train
# )
# class_weight_dict = dict(zip(classes, base_w))

# VAS_MULTIPLIER = 1.1   # try: 1.0, 1.2, 1.3, 1.5
# class_weight_dict["Vascular"] = class_weight_dict.get("Vascular", 1.0) * VAS_MULTIPLIER

# print("Final class weights:", class_weight_dict)

# # 2) Fit multinomial logistic regression
# clf_lr = Pipeline(steps=[
#     ("preprocess", clone(preprocess)),
#     ("model", LogisticRegression(
#         solver="lbfgs",          # supports multinomial automatically
#         max_iter=8000,
#         class_weight=class_weight_dict,
#         C=0.1
#     ))
# ])

# clf_lr.fit(X_train, y_train)

# # 3) Default prediction
# pred_lr = clf_lr.predict(X_test)

# # 4) OPTIONAL: vascular threshold override (often boosts vascular recall)
# USE_VASC_THRESHOLD = False
# T_VAS = 0.18  # try: 0.15–0.25

# if USE_VASC_THRESHOLD:
#     probs = clf_lr.predict_proba(X_test)
#     order = clf_lr.named_steps["model"].classes_
#     vas_idx = list(order).index("Vascular")
#     p_vas = probs[:, vas_idx]

#     pred_lr = pred_lr.copy()
#     pred_lr[p_vas >= T_VAS] = "Vascular"

# print("\n=== Logistic Regression (balanced weights + vascular boost + optional threshold) ===")
# print(classification_report(y_test, pred_lr, digits=4, labels=labels))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
# print(confusion_matrix(y_test, pred_lr, labels=labels))

# Section 3A: Logistic Regression (custom weights, C=0.1)

In [11]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.6380    0.7980    0.7091       614
NoDementia_Hy     0.9447    0.8791    0.9107      2217
       PD_LBD     0.5000    0.6102    0.5496        59
     Vascular     0.3810    0.2353    0.2909        34

     accuracy                         0.8492      2924
    macro avg     0.6159    0.6307    0.6151      2924
 weighted avg     0.8648    0.8492    0.8539      2924

Balanced accuracy: 0.6306562835490068
[[ 490  101   16    7]
 [ 244 1949   19    5]
 [  16    6   36    1]
 [  18    7    1    8]]


# Section 3G: Tuned Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

# 1) Created train/validation split from TRAIN only
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# 2) Fitted LR on inner-train

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_tr, y_tr)

# 3) Tuned vascular threshold on validation set
proba_val = clf_lr.predict_proba(X_val)
class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

thresholds = np.linspace(0.05, 0.50, 20)
results = []

for T in thresholds:
    preds = []
    for p in proba_val:
        if p[vas_idx] >= T:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    ba = balanced_accuracy_score(y_val, preds)

    # vascular recall
    yv = np.array(y_val)
    pv = np.array(preds)
    denom = np.sum(yv == "Vascular")
    vrec = (np.sum((yv == "Vascular") & (pv == "Vascular")) / denom) if denom else 0.0

    results.append((T, ba, vrec))

best_T, best_ba, best_vrec = max(results, key=lambda x: x[1])
print(f"Best T_VAS (by val BA): {best_T:.3f} | val BA={best_ba:.4f} | val VascRecall={best_vrec:.4f}")

# 4) Retrained LR on FULL TRAIN and evaluate once on TEST using tuned threshold
clf_lr.fit(X_train, y_train)

proba_test = clf_lr.predict_proba(X_test)
preds_test = []

for p in proba_test:
    if p[vas_idx] >= best_T:
        preds_test.append("Vascular")
    else:
        preds_test.append(class_order[np.argmax(p)])

print("\n=== Logistic Regression (Vascular threshold tuned) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, preds_test))
print(confusion_matrix(y_test, preds_test, labels=labels))

Best T_VAS (by val BA): 0.145 | val BA=0.6361 | val VascRecall=0.4444

=== Logistic Regression (Vascular threshold tuned) ===
               precision    recall  f1-score   support

           AD     0.6346    0.6221    0.6283       614
NoDementia_Hy     0.9490    0.8638    0.9044      2217
       PD_LBD     0.5556    0.5932    0.5738        59
     Vascular     0.0788    0.5588    0.1382        34

     accuracy                         0.8040      2924
    macro avg     0.5545    0.6595    0.5612      2924
 weighted avg     0.8649    0.8040    0.8308      2924

Balanced accuracy: 0.659493397063192
[[ 382   91   10  131]
 [ 199 1915   17   86]
 [  13    6   35    5]
 [   8    6    1   19]]


# Section 3E: HistGradientBoosting (tuned + weighted)
Balanced accuracy = 0.613
Vascular recall = 0.176
Worse than logistic regression.


In [13]:
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np

w_train = y_train.map({
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 10.0,
    "Vascular": 20.0
}).astype(float).values

clf_hgb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.03,
        max_depth=8,
        max_iter=1200,
        min_samples_leaf=10,
        l2_regularization=0.0,
        random_state=42
    ))
])

clf_hgb.fit(X_train, y_train, model__sample_weight=w_train)
pred_hgb = clf_hgb.predict(X_test)

print("\n=== HistGradientBoosting tuned (weighted) ===")
print(classification_report(y_test, pred_hgb, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_hgb))
print(confusion_matrix(y_test, pred_hgb, labels=labels))


=== HistGradientBoosting tuned (weighted) ===
               precision    recall  f1-score   support

           AD     0.6676    0.7622    0.7118       614
NoDementia_Hy     0.9384    0.8868    0.9119      2217
       PD_LBD     0.5068    0.6271    0.5606        59
     Vascular     0.1091    0.1765    0.1348        34

     accuracy                         0.8471      2924
    macro avg     0.5555    0.6131    0.5798      2924
 weighted avg     0.8632    0.8471    0.8537      2924

Balanced accuracy: 0.6131470395701917
[[ 468  112   16   18]
 [ 202 1966   19   30]
 [  14    7   37    1]
 [  17   10    1    6]]


# Section 3F: XGBoost (tuned + early stopping + strong class weighting)
Balanced accuracy = 0.569
Vascular recall = 0.029
Tree ensembles largely ignored vascular class despite weighting.


In [14]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]
label_to_int = {lab:i for i, lab in enumerate(labels)}
int_to_label = {i:lab for lab, i in label_to_int.items()}

y_train_int = y_train.map(label_to_int).astype(int)
y_test_int  = y_test.map(label_to_int).astype(int)

# stronger weights for tree models
TREE_WEIGHTS = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 10.0,
    "Vascular": 20.0
}
w_train = y_train.map(TREE_WEIGHTS).astype(float).values

# train/val split
X_tr, X_val, y_tr_int, y_val_int, w_tr, w_val = train_test_split(
    X_train, y_train_int, w_train,
    test_size=0.2, random_state=42, stratify=y_train_int
)

# preprocess
Xtr_mat = preprocess.fit_transform(X_tr)
Xval_mat = preprocess.transform(X_val)
Xte_mat  = preprocess.transform(X_test)

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(labels),

    n_estimators=5000,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=1,
    subsample=0.85,
    colsample_bytree=0.85,

    reg_lambda=1.0,
    gamma=0.0,

    tree_method="hist",
    random_state=42,
    eval_metric="mlogloss",

    early_stopping_rounds=200 
)

xgb.fit(
    Xtr_mat,
    y_tr_int,
    sample_weight=w_tr,
    eval_set=[(Xval_mat, y_val_int)],
    verbose=False
)

pred_int = xgb.predict(Xte_mat)
pred_xgb = pd.Series(pred_int).map(int_to_label)

print("\n=== XGBoost tuned (weighted + early stopping) ===")
print(classification_report(y_test, pred_xgb, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_xgb))
print(confusion_matrix(y_test, pred_xgb, labels=labels))


=== XGBoost tuned (weighted + early stopping) ===
               precision    recall  f1-score   support

           AD     0.6559    0.7638    0.7058       614
NoDementia_Hy     0.9333    0.8967    0.9147      2217
       PD_LBD     0.6071    0.5763    0.5913        59
     Vascular     0.0435    0.0294    0.0351        34

     accuracy                         0.8523      2924
    macro avg     0.5600    0.5666    0.5617      2924
 weighted avg     0.8582    0.8523    0.8540      2924

Balanced accuracy: 0.5665584653552209
[[ 469  119   11   15]
 [ 212 1988   10    7]
 [  17    8   34    0]
 [  17   15    1    1]]
